# Week 6 — Property features and school districts
Paired ablation: original, engineered without districts, engineered plus districts, for all three model families. Freeze Week 5 hyperparameters to isolate feature changes. April R² selects; May does not select features.

## 1. Feature definitions and prediction-time availability
PropertyAge = explicit valuation year 2026 minus YearBuilt (1600–2026); no CloseDate input. BedBathRatio = beds / positive baths; LivingAreaPerBedroom = positive area / positive beds. Log features use log1p with valid area/lot values. Zero denominators, invalid construction years and nonfinite inputs become missing for training-fitted imputation. Geographic interaction and squares center coordinates on fixed (37, -120), with no target-derived location aggregates. Original raw attributes are retained unchanged for comparability. Snapshot attributes may have been revised after listing; historical availability cannot be fully proven.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src import experiments
OUT = ROOT / 'outputs'
from src import features
import inspect
print(inspect.getsource(features.engineer))

## 2. Official school geography and cardinality audit
Public 2024–25 district polygons are downloaded separately; property coordinates never leave this computer. Native EPSG:3857 is recorded; download and point join use EPSG:4326. Elementary, secondary and unified IDs stay in separate columns. Multiple distinct IDs at the same level produce missing, not duplicate sales. Outside coverage/invalid points remain unmatched. No nearest-district invention. Only IDs enter the model, not demographic/enrollment fields. This is retrospective geography, not a verified historical boundary snapshot available before training.

In [ ]:
print((ROOT / 'data/external/school_districts/provenance.json').read_text())

## 3. Run paired feature experiments
Original rows reproduce Week 5 results. The six new feature-set fits use fixed per-family Week 5 winners and training-only preprocessing. Predictor counts distinguish input columns from transformed columns. Unknown districts stay missing through an explicit unknown category; they are never assigned a real district.

In [ ]:
comparison, may = experiments.run_week6(ROOT)
display(comparison)
print('Frozen Week 6 winner May:',may)

## 4. District coverage and paired improvement
School contribution is the same family with districts minus engineered without districts, on identical April rows. Validation deltas are descriptive, not causal or statistically proven.

In [ ]:
print((OUT / 'week06_spatial_audit.json').read_text())
display(pd.read_csv(OUT / 'week06_district_ablation.csv'))

## 5. Visual evidence
Train/April charts reveal overfitting; importance plots describe the engineered tree fits. May scatter/error plots describe the single selected frozen winner.

In [ ]:
for path in sorted((OUT / 'figures').glob('week06_*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## 6. Findings and limitations
See `docs/WEEK06_FINDINGS.md` and `docs/WEEKS_05_06_PRESENTATION_GUIDE.md`. Boundaries are 2024–25, transactions 2025–26, service edited in March 2026 and retrieved retrospectively. District membership may have changed; verify archival vintages before production or strict historical backtests. No school quality or causal effect is inferred. May was previously inspected; use newer unseen months for fresh confirmation.